In [ ]:
# Author: Niko Bleidistel
# last change: 2026-06-30

# Package Import

In [ ]:
from pathlib import Path 
from os import makedirs
import sys
import importlib
import re

import pandas as pd
import numpy as np

import mph
import matplotlib as mpl
import matplotlib.pyplot as plt

import time
import logging
import csv

In [ ]:
PYTHON_HELPER_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Python Helpers\COMSOL Python Helpers Version 1.0\py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import comsol_data_plotting as cdp
import plot_functions as pfs

# reload custom packages to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(cdp)
_ = importlib.reload(pfs)

# Constants

Hint: COMSOL Simulating and Plotting are independent. If you only want to plot the existing data you can turn all COMSOL SETTINGS to False.

In [ ]:
MFCO = False # False: use mf instead of mfco for COMSOL data extraction

## Simulation Settings 

In [ ]:
# COMSOL
START_SESSION = True       # is automatically skipped if COMSOL session is already running from previous runs 
END_SESSION = True         # end session after model is solved and data is exported (free up RAM and license)

# single mode 
SOLVE_MODEL = True         # solve the model before exporting data
EXPORT_DATA = True         # export data from the model 

# sweep mode 
SWEEP_MODEL = True          # Hint: always exports data

# Saving settings
SAVE_SMALL_MODEL_VERSION = True     # save a smaller version of the model (without solution) to reduce file size
SAVE_SOLVED_MODEL = True            # save full solved model (with solution) 

## Plotting Settings 

In [ ]:
TIME_ANALYSIS = True 

In [ ]:
# PLOTTING
PLOTALL = True             # plot all sweeps else only first found TXT file is plotted

IMPORT_DATA = True          # can be skipped if data is already loaded from previous runs
SAVE_PLOTS = True           # save plots to disk not only display them in the notebook


# Depth plots
MASK_AND_INTERPOLATE_DEPTH_DATA = True
DEPTH_PLOTS = True

# Homogeneity plots
MASK_AND_INTERPOLATE_HOMOGENEITY_DATA = True
HOMOGENEITY_PLOTS = True

# longitudinal plots
MASK_AND_INTERPOLATE_LONGITUNAL_DATA = True
LONGITUDINAL_PLOTS = True

## Fast Switch

In [ ]:
# overwrite
SIMULATION = False
PLOTTING = False

In [ ]:
if not SIMULATION:
    START_SESSION = False
    END_SESSION = False
    SOLVE_MODEL = False
    EXPORT_DATA = False
    SWEEP_MODEL = False
    SAVE_SMALL_MODEL_VERSION = False
    SAVE_SOLVED_MODEL = False

if not PLOTTING:
    PLOTALL = False             
    IMPORT_DATA = False
    SAVE_PLOTS = False
    MASK_AND_INTERPOLATE_DEPTH_DATA = False
    DEPTH_PLOTS = False
    MASK_AND_INTERPOLATE_HOMOGENEITY_DATA = False
    HOMOGENEITY_PLOTS = False
    MASK_AND_INTERPOLATE_LONGITUNAL_DATA = False
    LONGITUDINAL_PLOTS = False

## Paths

In [ ]:
MODELNAME_LIST = ["01_planar_waveguide_mf"] # Hint: currently only using the first model in the list
# INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\03_planar_waveguide")
# INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\04_simple_test_design_improved")
INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\05_planar_waveguide_mf")

OUTPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Script Outputs\000_New_Output")
makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

## Sweep Parameter

In [ ]:
if False:
    import numpy as np
SWEEP_PARAMETER = ["epilayer_xz_mesh_scale"]
SWEEP_VALUES = [
    [str(i.round(2)) for i in np.arange(5, 20.1, 1)],
    ]
print(f"SWEEP_VALUES: {SWEEP_VALUES}")

SWEEP_VALUES: [['5.0', '6.0', '7.0', '8.0', '9.0', '10.0', '11.0', '12.0', '13.0', '14.0', '15.0', '16.0', '17.0', '18.0', '19.0', '20.0']]


## Translations

In [ ]:
if MFCO:
    EXPORT_DICT = {
        "mfco.normB": "Magnetic flux density, norm [T]",
        "mfco.Bx": "Magnetic flux density, x-component [T]", 
        "mfco.By": "Magnetic flux density, y-component [T]", 
        "mfco.Bz": "Magnetic flux density, z-component [T]",
    }
    EXPORT_PARAMS = list(EXPORT_DICT.keys())
    EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

    # dictionary for translating parameter names to more descriptive labels for plotting
    TRANSLATE_PLOTLABELS = {
        "mfco.normB (T)": "Magnetic flux density, norm [T]",
        "mfco.Bx (T)": "Magnetic flux density, x-component [T]", 
        "mfco.By (T)": "Magnetic flux density, y-component [T]", 
        "mfco.Bz (T)": "Magnetic flux density, z-component [T]",
        "x": "longitudinal (x-axis) [m]",
        "y": "width (y-axis) [m]",
        "z": "depth (z-axis) [m]",
        "root.epilayer_mesh_scale": "epilayer mesh scale",
    }

    X_AXIS_PARAMS = ["z", "y"]
    Y_AXIS_PARAMS = ["mfco.normB (T)", "mfco.Bx (T)", "mfco.By (T)", "mfco.Bz (T)"]

else:
    EXPORT_DICT = {
        "mf.normB": "Magnetic flux density, norm [T]",
        "mf.Bx": "Magnetic flux density, x-component [T]", 
        "mf.By": "Magnetic flux density, y-component [T]", 
        "mf.Bz": "Magnetic flux density, z-component [T]",
    }
    EXPORT_PARAMS = list(EXPORT_DICT.keys())
    EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

    # dictionary for translating parameter names to more descriptive labels for plotting
    TRANSLATE_PLOTLABELS = {
        "mf.normB (T)": "Magnetic flux density, norm [T]",
        "mf.Bx (T)": "Magnetic flux density, x-component [T]", 
        "mf.By (T)": "Magnetic flux density, y-component [T]", 
        "mf.Bz (T)": "Magnetic flux density, z-component [T]",
        "x": "longitudinal (x-axis) [m]",
        "y": "width (y-axis) [m]",
        "z": "depth (z-axis) [m]",
        "root.epilayer_mesh_scale": "epilayer mesh scale",
    }

    X_AXIS_PARAMS = ["z", "y"]
    Y_AXIS_PARAMS = ["mf.normB (T)", "mf.Bx (T)", "mf.By (T)", "mf.Bz (T)"]

# Time LOG

In [ ]:
LOG_FILEPATH = str(OUTPUT_FOLDER / 'time_log.csv')

# Create the header only once; keep existing logs untouched.
log_path = Path(LOG_FILEPATH)
if (not log_path.exists()) or log_path.stat().st_size == 0:
    with open(LOG_FILEPATH, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow([
            'Timestamp', 'Log-Level', 'Message',
            'Since Start (s)', 'Since Start (H:M:S.ms)',
            'Since Last Log (s)', 'Since Last Log (H:M:S.ms)'
        ])

# Initialize logging in append mode
logging.basicConfig(
    filename=LOG_FILEPATH,
    filemode='a',
    level=logging.INFO,
    format='%(asctime)s;%(levelname)s;%(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True,
)

def format_hms_ms(seconds: float) -> str:
    """Converts seconds into HH:MM:SS.mmm format including milliseconds."""
    hms = time.strftime('%H:%M:%S', time.gmtime(seconds))
    milliseconds = int((seconds % 1) * 1000)
    return f"{hms}.{milliseconds:03d}"

def log_time(message: str, start_time: float, last_time: float) -> float:
    current_time = time.perf_counter()
    elapsed_since_start = current_time - start_time
    elapsed_since_last = current_time - last_time

    # Convert durations to HH:MM:SS.ms
    hms_start = format_hms_ms(elapsed_since_start)
    hms_last = format_hms_ms(elapsed_since_last)

    # Log all values separated by semicolons
    logging.info(
        f"{message};"
        f"{elapsed_since_start:.4f};{hms_start};"
        f"{elapsed_since_last:.4f};{hms_last}"
    )

    return current_time

In [ ]:
start_time = time.perf_counter()
last_time = start_time
last_time = log_time("Started running Python script", start_time, last_time)

# Simulation

In [ ]:
if START_SESSION:
    client, model_list = cde.import_models(
        client=None, 
        MODELNAME_LIST=MODELNAME_LIST, 
        printFeedback=True,
        avoid_reimporting=True,
        path_to_models=str(INPUT_FOLDER),
        )
    
    model = model_list[0]
    last_time = log_time("Imported models", start_time, last_time)


In [ ]:
if EXPORT_DATA:
    output_csv_path = OUTPUT_FOLDER / f"{model.name()}_parameters.csv"
    cde.save_Parameter_List_to_CSV(model, str(output_csv_path), displayParams=False)

    df_parameters = pd.read_csv(output_csv_path)
    EXPORT_PARAMS.extend([f"root.{param}" for param in df_parameters["name"].tolist()])
    EXPORT_DESCRIPTION.extend([f"{desc}" for desc in df_parameters["description"].tolist()])
    
    last_time = log_time("Exported model parameters to CSV", start_time, last_time)

In [ ]:
if START_SESSION:
    _ = cde.print_model_info(model)

In [ ]:
if START_SESSION:
    mph.tree(model)

In [ ]:
if SOLVE_MODEL:
    # model.build() # unnecessary if the model is not build: solve will build it automatically
    # model.mesh()  # unnecessary if the model is not meshed: solve will build it automatically
    model.solve()

    last_time = log_time("Solved model (single mode)", start_time, last_time)

In [ ]:
if SAVE_SOLVED_MODEL and SOLVE_MODEL:
    cde.save_as_copy(model, client, str(OUTPUT_FOLDER / f"{model.name()}_solved.mph"), smallerFilesize=False)
    last_time = log_time("Saved solved model (before data export)", start_time, last_time)

In [ ]:
if EXPORT_DATA and SOLVE_MODEL:
    cde.override_export_variables(model, "PyExport", EXPORT_PARAMS, "Study 1/Solution 1", EXPORT_DESCRIPTION)
    model.export("PyExport", OUTPUT_FOLDER / f"{model.name()}_exported_data.txt")
    
    last_time = log_time("Exported solution data", start_time, last_time)

In [ ]:
if SAVE_SOLVED_MODEL and SOLVE_MODEL:
    cde.save_as_copy(model, client, str(OUTPUT_FOLDER / f"{model.name()}_solved.mph"), smallerFilesize=False)
    last_time = log_time("Saved solved model", start_time, last_time)

In [ ]:
if SAVE_SMALL_MODEL_VERSION:
    cde.save_as_copy(model, client, str(OUTPUT_FOLDER / f"{model.name()}_smallfile.mph"), smallerFilesize=True)
    last_time = log_time("Saved model with reduced file size", start_time, last_time)

In [ ]:
if SWEEP_MODEL:
    cde.parameter_sweep(
        model = model,
        client = client,
        studyname = "Study 1",
        parameternames = SWEEP_PARAMETER,
        parametervalues = SWEEP_VALUES,
        expressions = EXPORT_PARAMS,
        export_path = str(OUTPUT_FOLDER/"Sweep_Data"),
        dataset_identifier = "Study 1/Solution 1",
        markwithParameters = [SWEEP_PARAMETER[0]],
        saveSolutions = SAVE_SOLVED_MODEL,
    )
    last_time = log_time("Finished parameter sweep", start_time, last_time)

In [ ]:
if END_SESSION:
    # Attention: Following will lose all unsaved changes (and solutions)
    client.clear() # explicitly clear the client from RAM (unnecessary for "multi='off", but good practice)
    client.disconnect() # disconnect the client from the COMSOL server -> free up the license 
    last_time = log_time("Ended COMSOL session", start_time, last_time)

# Time Analysis

In [ ]:
if TIME_ANALYSIS:
    # import the time log CSV file into a pandas DataFrame
    df_time_log = pd.read_csv(LOG_FILEPATH, sep=";")
    df_time_log["Timestamp"] = pd.to_datetime(df_time_log["Timestamp"])
    display(df_time_log.head(20))

In [ ]:
if TIME_ANALYSIS:
    # get start and end timestamps for each study
    starts = df_time_log[df_time_log["Message"].str.contains("Running study")].reset_index(drop=True)
    ends = df_time_log[df_time_log["Message"].str.contains("Finished solving study")].reset_index(drop=True)

    # check if the number of starts and ends match
    min_len = min(len(starts), len(ends))
    if len(ends) > len(starts):
        print(f"Warning: {len(ends) - len(starts)} studies have end timestamps but no start timestamps.")

    number_of_missing_ends = len(starts) - len(ends)
    # print(f"Number of studies with missing end timestamps: {number_of_missing_ends}")

    if number_of_missing_ends > 0:
        print(f"Warning: {number_of_missing_ends} studies are missing end timestamps. Using {min_len} pairs.")

    # calculate durations
    durations = ends["Timestamp"][:min_len] - starts["Timestamp"][:min_len]

    # create a new DataFrame with the results
    df_durations = pd.DataFrame(
        {
            "Study_Start": starts["Timestamp"][:min_len],
            "Study_End": ends["Timestamp"][:min_len],
            "Duration": durations,
            "Duration_Seconds": durations.dt.total_seconds(),
        }
    )
    display(df_durations)

In [ ]:
if TIME_ANALYSIS:
    # Build parameter table directly from SWEEP_PARAMETER and SWEEP_VALUES
    if len(SWEEP_PARAMETER) != len(SWEEP_VALUES):
        print(
            f"Warning: SWEEP_PARAMETER has {len(SWEEP_PARAMETER)} entries, "
            f"but SWEEP_VALUES has {len(SWEEP_VALUES)} entries. Using only matching pairs."
        )

    sweep_lengths = [len(values) for values in SWEEP_VALUES]
    expected_iterations = max(sweep_lengths) if len(sweep_lengths) > 0 else 0

    if len(set(sweep_lengths)) > 1:
        print(
            f"Warning: SWEEP_VALUES entries have different lengths ({sweep_lengths}). "
            f"Missing values will be filled with NaN in df_params."
        )

    # Keep only the latest expected sweep iterations in the duration table
    target_duration_rows = max(expected_iterations - max(number_of_missing_ends, 0), 0)
    if target_duration_rows > 0:
        if len(df_durations) != target_duration_rows:
            print(
                f"Warning: Number of expected sweep iterations ({target_duration_rows}) and duration entries ({len(df_durations)}) do not match. "
                f"Keeping only the last {target_duration_rows} entries of df_durations."
            )
        df_durations = df_durations.iloc[-target_duration_rows:].reset_index(drop=True)
    else:
        print("Warning: No valid sweep iteration count found from SWEEP_VALUES.")

    display(df_durations)

In [ ]:
if TIME_ANALYSIS:
    # Create parameter dataframe indexed by Iteration
    params_dict = {}
    for param_name, param_values in zip(SWEEP_PARAMETER, SWEEP_VALUES):
        series = pd.Series(param_values)
        # Try numeric conversion, keep original text if conversion fails
        numeric_series = pd.to_numeric(series, errors="coerce")
        if numeric_series.notna().all():
            params_dict[param_name] = numeric_series
        else:
            params_dict[param_name] = series

    if len(params_dict) > 0:
        df_params = pd.DataFrame(params_dict)
        df_params.index = pd.RangeIndex(start=1, stop=len(df_params) + 1, name="Iteration")
    else:
        df_params = pd.DataFrame(index=pd.RangeIndex(start=1, stop=1, name="Iteration"))

    # remove the last number_of_missing_ends rows from df_params if there are missing ends
    if number_of_missing_ends > 0:
        df_params = df_params.iloc[:-number_of_missing_ends]
        
    display(df_params)

In [ ]:
if TIME_ANALYSIS:
    # Optional combined overview: sweep parameters + durations per iteration
    if len(df_durations) > 0 and len(df_params) > 0:
        df_summary = df_params.copy()
        duration_subset = df_durations[["Duration", "Duration_Seconds"]].reset_index(drop=True)
        duration_subset.index = pd.RangeIndex(start=1, stop=len(duration_subset) + 1)
        df_summary = df_summary.join(duration_subset, how="left")
        display(df_summary)

In [ ]:
if TIME_ANALYSIS:
    # calculate the step sizes (differences) for each parameter
    df_diffs = df_params.diff().dropna()

    constant_step_sizes = {}
    for col in df_params.columns:
        # check if step size is constant
        if np.all(np.isclose(df_diffs[col], df_diffs[col].iloc[0])):
            constant_step_sizes[col] = df_diffs[col].iloc[0]
        # else:
        #     constant_step_sizes[col] = np.nan
    df_step_sizes = pd.DataFrame(list(constant_step_sizes.items()), columns=['Parameter', 'Step Size'])
    df_step_sizes.replace([np.nan], ["Not Constant"], inplace=True) 
    display(df_step_sizes)

In [ ]:
if TIME_ANALYSIS:
    lines_data = []
    max_col_len = 0
    max_min_len = 0
    max_max_len = 0

    for col in df_params.columns:
        min_val = df_params[col].min()
        max_val = df_params[col].max()
        
        if min_val < 1e3 or min_val > 1 or max_val < 1e3 or max_val > 1:
            min_str = f"{min_val:.2f}"
            max_str = f"{max_val:.2f}"
        else:
            min_str = f"{min_val:.2e}"
            max_str = f"{max_val:.2e}"
            
        # Check for optional step size
        step_str = ""
        if col in df_step_sizes['Parameter'].values:
            step_size = df_step_sizes[df_step_sizes['Parameter'] == col]['Step Size'].values[0]
            step_str = f"with step size {step_size}"
            
        clean_col = col.replace("_", " ")
        
        # Track maximum lengths for alignment
        max_col_len = max(max_col_len, len(clean_col))
        max_min_len = max(max_min_len, len(min_str))
        max_max_len = max(max_max_len, len(max_str))
            
        lines_data.append((clean_col, min_str, max_str, step_str))

    # Build lines with precise column alignment
    final_lines = []
    for c, mn, mx, st in lines_data:
        # 1. Align column name (right-aligned)
        line = f"{c.rjust(max_col_len)} = "
        
        if mn == mx:
            # Single value case: left-align and pad to match the full "to max_val" width
            total_range_width = max_min_len + len(" to ") + max_max_len
            line += f"{mn.ljust(total_range_width)}"
        else:
            # Range case: left-align both values in their respective columns
            line += f"{mn.ljust(max_min_len)} to {mx.ljust(max_max_len)}"
            
        # 2. Append step size if it exists
        if st:
            line += f" {st}"
            
        final_lines.append(line.rstrip())

    # Determine total width based on the longest line to center the header
    content_width = max(len(line) for line in final_lines)
    header_text = "Sweep Duration Analysis"

    # Center the header; use fallback if header is wider than content
    header_padding = max(0, content_width)
    header_line = header_text.center(header_padding)

    # Combine header and parameters with a blank line for visual spacing
    plot_title = f"{header_line}\n\n" + "\n".join(final_lines)
    print(plot_title)

In [ ]:
if TIME_ANALYSIS:
    fig, ax =pfs.u_plot_scatter_with_error_bars(
        df_durations.index.tolist(),
        df_durations["Duration_Seconds"].tolist(),
        x_label="Interation Index",
        y_label="Duration [s]", 
        title=plot_title,
        fix_title_spacing=True,
        )
    fig.savefig(fname=str(OUTPUT_FOLDER / "study_duration_analysis.png"), dpi=300, bbox_inches="tight")

# Plotting

## Plot All

In [ ]:
# test area for depth y-sweep
if False:
    import numpy as np

    epilayer_width = 15e-6

    depth_ygrid = [f"{i:.2e}" for i in np.arange(0, epilayer_width, 3e-6)]

    print(depth_ygrid)
    print(len(depth_ygrid))

In [ ]:
def sweep_plot_workflow(txt_path: str, last_time: float):
    last_time = log_time(f"Started plotting workflow for sweep results in {Path(txt_path).name}", start_time, last_time)

    output_folder = Path(txt_path).parent

    print(f"\nProcessing file: {txt_path}...")

    if IMPORT_DATA:
        # import data from txt file
        header_data, df = cdp.read_comsol_export(txt_path)

        # import parameter data from CSV file
        modelname = str(header_data.get('Model')).replace(".mph", "")
        
        print("Data imported successfully.")
        last_time = log_time(f"Imported data from {Path(txt_path).name}", start_time, last_time)

    if MASK_AND_INTERPOLATE_DEPTH_DATA:
        # define grid points for masking and interpolation
        epilayer_height = df['root.epilayer_height (m)'].iloc[0]
        insulator_height = df['root.insulator_height (m)'].iloc[0]

        DEPTH_GRID_DICT = {
            'x': [0.0],
            'y': [0.0],
            }
        
        limit_dict = {
            'z': [0, - epilayer_height] - insulator_height,
            }
        
        
        print("Now masking and interpolating data for depth plots...")
        

        # mask and interpolate data for depth plots
        df_depth = cdp.mask_and_interpolate_data(
            df = df,
            x_params = X_AXIS_PARAMS,
            y_params = Y_AXIS_PARAMS,
            grid_dict = DEPTH_GRID_DICT,
            limit_dict = limit_dict,
            )
        
        last_time = log_time(f"Masked and interpolated depth data for {Path(txt_path).name}", start_time, last_time)

    if DEPTH_PLOTS:
        for y_param in Y_AXIS_PARAMS:
            print(f"\nPlotting {y_param}...")

            fig, ax = cdp.standard_plot(
                output_folder = None,
                x_param = "z",
                y_param = y_param,
                header_data = header_data,
                df_curves = df_depth,
                df_param = df,
                title_params = [f"root.{param}" for param in SWEEP_PARAMETER],
                sweep_params = None,
                title = None,
                custom_label = None,
                translation_dict = TRANSLATE_PLOTLABELS,
                fig = None, 
                ax = None,
                color = 'midnightblue',
                save_plot = False,
                )
            last_time = log_time(f"Plotted {y_param} over z (depth)", start_time, last_time)
                
            if SAVE_PLOTS:
                plot_output_folder = str(output_folder / Path("Plots/Depth"))
                makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist
                
                modelname = str(header_data.get('Model')).replace(".mph", "")
                output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_depth.png"
                fig.savefig(str(output_path), dpi=300) # type: ignore
                
                last_time = log_time(f"Saved {y_param} depth plot", start_time, last_time)

            plt.close(fig)



    if MASK_AND_INTERPOLATE_HOMOGENEITY_DATA:
        # define grid points for masking and interpolation
        
        insulator_height = df['root.insulator_height (m)'].iloc[0]
        epilayer_width = df['root.epilayer_width (m)'].iloc[0]

        GRID_DICT = {
            'x': [0.0],
            'z': [-5e-6, -6e-6, -7e-6, -8e-6, -9e-6, -1e-5]- insulator_height,
            }
        limit_dict = {
            'y': [-epilayer_width/2, epilayer_width/2]
            }

        print("Now masking and interpolating data for homogeneity plots...")
        last_time = log_time(f"Masked and interpolated homogeneity data for {Path(txt_path).name}", start_time, last_time)
        # mask and interpolate data for homogeneity plots
        df_homogeneity = cdp.mask_and_interpolate_data(
            df = df,
            x_params = X_AXIS_PARAMS,
            y_params = Y_AXIS_PARAMS,
            grid_dict = GRID_DICT,
            limit_dict = limit_dict,
            )


    if HOMOGENEITY_PLOTS:
        for y_param in Y_AXIS_PARAMS:
            print(f"\nPlotting {y_param}...")

            z_values = list(GRID_DICT['z'])
            # cmap = mpl.colormaps['magma'] # type: ignore
            cmap = mpl.colormaps['viridis'] # type: ignore


            fig, ax = None, None  
            for idx, z in enumerate(z_values):

                print(f"Plotting curve for z = {z} m...")
                df_homogeneity_z = df_homogeneity[df_homogeneity['z'] == z]

                color = cmap(idx / len(z_values))  # get color from colormap based on index
            
                fig, ax = cdp.standard_plot(
                    output_folder = None,
                    x_param = "y",
                    y_param = y_param,
                    header_data = header_data,
                    df_curves = df_homogeneity_z,
                    df_param = df,
                    title_params = [f"root.{param}" for param in SWEEP_PARAMETER],
                    sweep_params = ["z"],
                    translation_dict = TRANSLATE_PLOTLABELS,
                    fig = fig, 
                    ax = ax,
                    color=color,
                    save_plot = False,
                    )
                last_time = log_time(f"Plotted {y_param} over y (homogeneity)", start_time, last_time)

            if SAVE_PLOTS:
                plot_output_folder = str(output_folder / Path("Plots/Homogeneity"))
                makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist
                
                modelname = str(header_data.get('Model')).replace(".mph", "")
                output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_homogeneity.png"
                fig.savefig(str(output_path), dpi=300) # type: ignore
                
                last_time = log_time(f"Saved {y_param} homogeneity plot", start_time, last_time)
            plt.close(fig)

    if MASK_AND_INTERPOLATE_LONGITUNAL_DATA:
        # define grid points for masking and interpolation    
        insulator_height = df['root.insulator_height (m)'].iloc[0]
        epilayer_length = df['root.epilayer_length (m)'].iloc[0]

        GRID_DICT = {
            'y': [0.0],
            'z': [-5e-6, -6e-6, -7e-6, -8e-6, -9e-6, -1e-5]- insulator_height,
            }
        limit_dict = {
            'x': [-epilayer_length/2, epilayer_length/2]
            }

        # mask and interpolate data for longitudinal plots
        df_longitudinal = cdp.mask_and_interpolate_data(
            df = df,
            x_params = X_AXIS_PARAMS,
            y_params = Y_AXIS_PARAMS,
            grid_dict = GRID_DICT,
            limit_dict = limit_dict,
            )
        last_time = log_time(f"Masked and interpolated longitudinal data", start_time, last_time)

    if LONGITUDINAL_PLOTS:
        for y_param in Y_AXIS_PARAMS:
            print(f"\nPlotting {y_param}...")

            z_values = list(GRID_DICT['z'])
            # cmap = mpl.colormaps['magma'] # type: ignore
            cmap = mpl.colormaps['viridis'] # type: ignore


            fig, ax = None, None  
            for idx, z in enumerate(z_values):

                print(f"Plotting curve for z = {z} m...")
                df_longitudinal_z = df_longitudinal[df_longitudinal['z'] == z]

                color = cmap(idx / len(z_values))  # get color from colormap based on index

                fig, ax = cdp.standard_plot(
                    output_folder = None,
                    x_param = "x",
                    y_param = y_param,
                    header_data = header_data,
                    df_curves = df_longitudinal_z,
                    df_param = df,
                    title_params = [f"root.{param}" for param in SWEEP_PARAMETER],
                    sweep_params = ["z"],
                    translation_dict = TRANSLATE_PLOTLABELS,
                    fig = fig, 
                    ax = ax,
                    color=color,
                    save_plot = False,
                    )
                last_time = log_time(f"Plotted {y_param} over x (longitudinal)", start_time, last_time)

            if SAVE_PLOTS:
                plot_output_folder = str(output_folder / Path("Plots/Longitudinal"))
                makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist

                modelname = str(header_data.get('Model')).replace(".mph", "")
                output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_longitudinal.png"
                fig.savefig(str(output_path), dpi=300) # type: ignore

                last_time = log_time(f"Saved {y_param} longitudinal plot", start_time, last_time)
            plt.close(fig)
            
    plt.close('all')
    return last_time

In [ ]:
txt_paths =cdp.find_all_files_in_folder(file_extension="txt", folder_path=str(OUTPUT_FOLDER))
if len(txt_paths) == 0:
    print("No TXT files found in the output folder.")
    print(OUTPUT_FOLDER)
else:
    # sort the txt_paths naturally (e.g., "file2.txt" comes before "file10.txt")
    txt_paths = sorted(txt_paths, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))])
    
    print(f"Found {len(txt_paths)} TXT files in the output folder:")
    print(str(OUTPUT_FOLDER))
    for txt_path in txt_paths:
        relative_path = Path(txt_path).relative_to(OUTPUT_FOLDER)
        print(f"-> {relative_path}")

Found 16 TXT files in the output folder:
R:\Bleidistel_Niko\COMSOL\COMSOL Script Outputs\000_New_Output
-> 01_planar_waveguide_mf_exported_data.txt
-> Sweep_Data\Iteration 0 (epilayer_xz_mesh_scale 5.0)\01_planar_waveguide_mf_iteration_0.txt
-> Sweep_Data\Iteration 1 (epilayer_xz_mesh_scale 6.0)\01_planar_waveguide_mf_iteration_1.txt
-> Sweep_Data\Iteration 2 (epilayer_xz_mesh_scale 7.0)\01_planar_waveguide_mf_iteration_2.txt
-> Sweep_Data\Iteration 3 (epilayer_xz_mesh_scale 8.0)\01_planar_waveguide_mf_iteration_3.txt
-> Sweep_Data\Iteration 4 (epilayer_xz_mesh_scale 9.0)\01_planar_waveguide_mf_iteration_4.txt
-> Sweep_Data\Iteration 5 (epilayer_xz_mesh_scale 10.0)\01_planar_waveguide_mf_iteration_5.txt
-> Sweep_Data\Iteration 6 (epilayer_xz_mesh_scale 11.0)\01_planar_waveguide_mf_iteration_6.txt
-> Sweep_Data\Iteration 7 (epilayer_xz_mesh_scale 12.0)\01_planar_waveguide_mf_iteration_7.txt
-> Sweep_Data\Iteration 8 (epilayer_xz_mesh_scale 13.0)\01_planar_waveguide_mf_iteration_8.txt
->

In [ ]:
if PLOTALL:
    last_time = log_time(f"Starting plotting all available txts", start_time, last_time)
    for txt_path in txt_paths:
        if "errormessage.txt" in Path(txt_path).name:
            print(f"Skipping error message file: {txt_path}")
            last_time = log_time(f"Skipped error message file: {txt_path}", start_time, last_time)
            continue
        else:
            last_time = sweep_plot_workflow(txt_path, last_time)
            last_time = log_time(f"Finished plotting workflow for {txt_path}", start_time, last_time)
    plt.close('all')

## Plot Single

In [ ]:
if IMPORT_DATA and not PLOTALL:
    # import data from txt file
    header_data, df = cdp.read_comsol_export(str(txt_paths[0]))
    display(header_data)
    display(df.head())

    # import parameter data from CSV file
    modelname = str(header_data.get('Model')).replace(".mph", "") 
    output_csv_path = OUTPUT_FOLDER / f"{modelname}_parameters.csv"

    last_time = log_time(f"Imported parameter data from {txt_paths[0]}", start_time, last_time)

In [ ]:
if MASK_AND_INTERPOLATE_DEPTH_DATA and not PLOTALL:
    # define grid points for masking and interpolation
    DEPTH_GRID_DICT = {
        'x': [0.0],
        'y': [0.0],
        }
    epilayer_height = df['root.epilayer_height (m)'].iloc[0]
    insulator_height = df['root.insulator_height (m)'].iloc[0]
    limit_dict = {
        'z': [0, - epilayer_height] - insulator_height,
        }
    display(limit_dict)

    # mask and interpolate data for depth plots
    df_depth = cdp.mask_and_interpolate_data(
        df = df,
        x_params = X_AXIS_PARAMS,
        y_params = Y_AXIS_PARAMS,
        grid_dict = DEPTH_GRID_DICT,
        limit_dict = limit_dict,
        )
    last_time = log_time(f"Masked and interpolated depth data", start_time, last_time)

In [ ]:
if DEPTH_PLOTS and not PLOTALL:
    for y_param in Y_AXIS_PARAMS:
        print(f"\nPlotting {y_param}...")

        fig, ax = cdp.standard_plot(
            output_folder = None,
            x_param = "z",
            y_param = y_param,
            header_data = header_data,
            df_curves = df_depth,
            df_param = df,
            title_params = None,
            sweep_params = None,
            translation_dict = TRANSLATE_PLOTLABELS,
            fig = None, 
            ax = None,
            color = 'midnightblue',
            save_plot = False,
            )
        
        last_time = log_time(f"Plotted {y_param} over z (depth)", start_time, last_time)
            
        if SAVE_PLOTS:
            plot_output_folder = str(OUTPUT_FOLDER / Path("Plots/Depth"))
            makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist
            
            modelname = str(header_data.get('Model')).replace(".mph", "")
            output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_depth.png"
            fig.savefig(str(output_path), dpi=300) # type: ignore
            last_time = log_time(f"Saved {y_param} depth plot", start_time, last_time)

In [ ]:
if MASK_AND_INTERPOLATE_HOMOGENEITY_DATA and not PLOTALL:
    # define grid points for masking and interpolation    
    insulator_height = df['root.insulator_height (m)'].iloc[0]
    epilayer_width = df['root.epilayer_width (m)'].iloc[0]

    GRID_DICT = {
        'x': [0.0],
        'z': [-5e-6, -6e-6, -7e-6, -8e-6, -9e-6, -1e-5]- insulator_height,
        }
    limit_dict = {
        'y': [-epilayer_width/2, epilayer_width/2]
        }
    display(limit_dict)

    # mask and interpolate data for homogeneity plots
    df_homogeneity = cdp.mask_and_interpolate_data(
        df = df,
        x_params = X_AXIS_PARAMS,
        y_params = Y_AXIS_PARAMS,
        grid_dict = GRID_DICT,
        limit_dict = limit_dict,
        )
    last_time = log_time(f"Masked and interpolated homogeneity data", start_time, last_time)

In [ ]:
if HOMOGENEITY_PLOTS and not PLOTALL:
    for y_param in Y_AXIS_PARAMS:
        print(f"\nPlotting {y_param}...")

        z_values = list(GRID_DICT['z'])
        # cmap = mpl.colormaps['magma'] # type: ignore
        cmap = mpl.colormaps['viridis'] # type: ignore


        fig, ax = None, None  
        for idx, z in enumerate(z_values):

            print(f"Plotting curve for z = {z} m...")
            df_homogeneity_z = df_homogeneity[df_homogeneity['z'] == z]

            color = cmap(idx / len(z_values))  # get color from colormap based on index
        
            fig, ax = cdp.standard_plot(
                output_folder = None,
                x_param = "y",
                y_param = y_param,
                header_data = header_data,
                df_curves = df_homogeneity_z,
                df_param = df,
                title_params = None,
                sweep_params = ["z"],
                translation_dict = TRANSLATE_PLOTLABELS,
                fig = fig, 
                ax = ax,
                color=color,
                save_plot = False,
                )
            last_time = log_time(f"Plotted {y_param} over y (homogeneity)", start_time, last_time)
            
        if SAVE_PLOTS:
            plot_output_folder = str(OUTPUT_FOLDER / Path("Plots/Homogeneity"))
            makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist
            
            modelname = str(header_data.get('Model')).replace(".mph", "")
            output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_homogeneity.png"
            fig.savefig(str(output_path), dpi=300) # type: ignore
            
            last_time = log_time(f"Saved {y_param} homogeneity plot", start_time, last_time)

In [ ]:
if MASK_AND_INTERPOLATE_LONGITUNAL_DATA and not PLOTALL:
    # define grid points for masking and interpolation    
    insulator_height = df['root.insulator_height (m)'].iloc[0]
    epilayer_length = df['root.epilayer_length (m)'].iloc[0]

    GRID_DICT = {
        'y': [0.0],
        'z': [-5e-6, -6e-6, -7e-6, -8e-6, -9e-6, -1e-5]- insulator_height,
        }
    limit_dict = {
        'x': [-epilayer_length/2, epilayer_length/2]
        }

    # mask and interpolate data for longitudinal plots
    df_longitudinal = cdp.mask_and_interpolate_data(
        df = df,
        x_params = X_AXIS_PARAMS,
        y_params = Y_AXIS_PARAMS,
        grid_dict = GRID_DICT,
        limit_dict = limit_dict,
        )
    last_time = log_time(f"Masked and interpolated longitudinal data", start_time, last_time)

In [ ]:
if LONGITUDINAL_PLOTS and not PLOTALL:
    for y_param in Y_AXIS_PARAMS:
        print(f"\nPlotting {y_param}...")

        z_values = list(GRID_DICT['z'])
        # cmap = mpl.colormaps['magma'] # type: ignore
        cmap = mpl.colormaps['viridis'] # type: ignore


        fig, ax = None, None  
        for idx, z in enumerate(z_values):

            print(f"Plotting curve for z = {z} m...")
            df_longitudinal_z = df_longitudinal[df_longitudinal['z'] == z]

            color = cmap(idx / len(z_values))  # get color from colormap based on index
        
            fig, ax = cdp.standard_plot(
                output_folder = None,
                x_param = "x",
                y_param = y_param,
                header_data = header_data,
                df_curves = df_longitudinal_z,
                df_param = df,
                title_params = None,
                sweep_params = ["z"],
                translation_dict = TRANSLATE_PLOTLABELS,
                fig = fig, 
                ax = ax,
                color=color,
                save_plot = False,
                )
            last_time = log_time(f"Plotted {y_param} over x (longitudinal)", start_time, last_time)
            
        if SAVE_PLOTS:
            plot_output_folder = str(OUTPUT_FOLDER / Path("Plots/Longitudinal"))
            makedirs(plot_output_folder, exist_ok=True)  # create output folder if it doesn't exist
            
            modelname = str(header_data.get('Model')).replace(".mph", "")
            output_path = Path(plot_output_folder) / f"{modelname}_{y_param}_longitudinal.png"
            fig.savefig(str(output_path), dpi=300) # type: ignore
            
            last_time = log_time(f"Saved {y_param} longitudinal plot", start_time, last_time)

# End